# 11 · Logistic regression for future prostate IHC (and NE-positive IHC) on the risk score

Does the frozen federated (no-DFCI) **platinum-trained XGBoost** score, at each ADT landmark, predict a future
immunohistochemistry (IHC) workup of a prostate-cancer specimen, and whether that IHC shows NEPC / small cell /
neuroendocrine differentiation? Each outcome is an unpenalized logistic regression on the score alone
(`outcome ~ RISK_Z`, z-scored within each fit). All logic lives in `COMPASS/multivariable_models/ihc_lr.py`.

| Outcome | Event | Population (per landmark) |
|---|---|---|
| `ihc_any` | ≥1 qualifying IHC note after ADT + landmark | every at-risk patient |
| `ne_pos_all` | ≥1 NE-positive qualifying IHC note after ADT + landmark | at-risk patients with complete LLM review and no NE-positive IHC note at or before ADT + landmark |
| `ne_pos_given_ihc` | same as `ne_pos_all` | the `ne_pos_all` patients who have ≥1 qualifying future IHC note |

**At risk.** The `endpoint == "nepc"` rows of 04's `transfer_scores.csv` (XGBoost, `config == "both"`). Patients
with LLM-NEPC or censoring at or before the landmark are not scored there, so they are not in any fit.

**IHC notes (regex, pathology reports only).** A report has IHC when the raw `RPT_TEXT` has either:
- a generic IHC term (immunohistochemistry, IHC, immunostain, immunoperoxidase, immunoreactive) that is not
  negated or unresulted ("no IHC performed", "pending", "recommended"); or
- a named marker from any panel:
  - NE: synaptophysin, chromogranin, CD56, INSM1, NSE, taken from LCA's `ne_gate.NE_MARKER`;
  - lineage: NKX3.1, AR, PSA, PSMA, PSAP, prostein;
  - basal / diagnostic: p63, HMWCK, CK5/6, AMACR, PIN4;
  - other: ERG, Ki-67, TTF-1, GATA3, CK7, CK20, p53, RB1, PTEN.

AR, PSA, PSMA, ERG, RB1 and PTEN count only when a stain cue sits in the same clause. Serum PSA values and
PSMA PET mentions are never counted.

A note **qualifies** when it has IHC and either of these holds:
- the first site named in `PROC_DESC` plus the first 300 characters is the prostate;
- the text states prostatic origin.

A note that names a non-prostate primary is excluded, unless it also states prostatic origin.

**NE-positive.** A qualifying note counts as NE-positive when the same report (`DFCI_MRN` + `RPT_ID`) has ≥1 LLM NEPC
finding, of any evidence type, that passes the gate `nepc_labels.parquet` uses (ne_gate v3, the trial-context screen,
and final attempts only). The gate is imported read-only from `LLM_clinical_annotations`, and its commit is recorded
in `run_meta.json`. A note without a gated finding is NE-negative only when the patient has complete LLM review
(`ok`) or validated `no_trigger_notes` status. Patients with failed, not-run, partial, or missing review
status are excluded from both NE outcomes, even when some positive findings are already known;
they remain eligible for `ihc_any`. Counts report `n_ne_review_complete` and `n_ne_review_excluded`.
Ambiguous report IDs, including LCA collision suffixes, stop the analysis for identity reconciliation.

**Timing.** Notes are dated by `EVENT_DATE`. "Future" means ADT + landmark < note date ≤ ADT + 3650 d. The label is
"ever in that window", not a fixed horizon. Prior IHC excludes no one. Every scored patient must have an ADT anchor; missing anchors stop the analysis.

**Statistics.**
- AUC is out-of-fold (stratified 5-fold) with a patient-bootstrap 95% CI.
- The raw-score AUC (below) is reported alongside it.
- ORs (per SD), Wald CIs and LRT p come from the full-population fit.
- Fewer than `MIN_EVENTS` events or non-events gives a count-only row.

**Raw-score AUC.** `auc_raw_score` is the AUC of the untransformed frozen score itself, with no fitting,
no CV and no sign flip, and a patient-bootstrap 95% CI drawn with the same seed (so the resamples are paired with
the OOF CI). It does not depend on CV, so failed folds never suppress it. For this external, frozen score it is the
preferred discrimination measure. With few events and a weak or null predictor, the OOF AUC is biased below 0.5:
each fold's refitted intercept shifts that fold's predictions, and those shifts can dominate the ranking.
The risk-only LR is monotone in the score, so its full-population AUC equals the raw-score AUC whenever OR > 1.

**Out-of-fold predictions.** `lr_oof_predictions.csv` holds one probability per patient for each outcome × landmark,
covering every patient in that outcome's population. Each probability comes from the CV fit that held that patient
out, the same predictions the AUC uses. `event` is that outcome's label. Each outcome × landmark is its own fit, so
do not pool rows across them. The results report a Brier score, calibration intercept and slope, and observed vs
predicted by quintile.

**Outputs.** These go under the protected data root `DATA_ROOT/ihc_lr/`:
- aggregate CSVs;
- `lr_oof_predictions.csv` (patient level);
- a note-level audit parquet (tags only, no report text);
- `run_meta.json`.

**Prerequisites.** Notebook 04 has written `transfer_scores.csv`. The LLM NEPC extraction has run. The LCA repo is
checked out at `COMPASS_LCA_ROOT`.

In [ ]:
import json, os, sys
sys.path.insert(0, "../../pipeline")
import compass_pipeline as cp
for _p in (str(cp.PROJECT_ROOT),):
    if _p not in sys.path: sys.path.insert(0, _p)
from pathlib import Path
import pandas as pd
from IPython.display import display
from COMPASS.multivariable_models import ihc_lr as il
nsl = il.nsl

DATA_ROOT  = Path(cp._PROFILE_OUTPUT_ROOT)
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))

# >>> Edit <<<
COHORTS     = ("all",)          # "all" = every patient 04 scored
LANDMARKS   = (0, 90, 180)
MIN_EVENTS  = 5
N_FOLDS     = 5
N_BOOTSTRAP = 200
SEED        = 0

SCORES_PATH    = RHINO_ROOT / "federated_transfer_adt_no_DFCI" / "all_models" / "transfer_scores.csv"
DATA_PATH      = DATA_ROOT / "longitudinal_prediction_data_adt.csv"      # ADT anchor dates
PATHOLOGY_PATH = Path(os.environ.get("PROFILE_PATHOLOGY_NOTES",
                                     str(Path(cp._PROFILE_DATA_ROOT) / "CLINICAL_NOTES" / "PATHOLOGY_NOTES.parquet")))
LLM_NEPC_DIR   = Path("/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_nepc")
# LLM_clinical_annotations: a sibling of the project (cluster: /data/gusev/USERS/jpconnor/code/), else one level up (local).
_LCA_CANDIDATES = [Path(cp.PROJECT_ROOT).parent / "LLM_clinical_annotations",
                   Path(cp.PROJECT_ROOT).parent.parent / "LLM_clinical_annotations"]
LCA_ROOT       = Path(os.environ.get("COMPASS_LCA_ROOT",
                                     str(next((p for p in _LCA_CANDIDATES if p.exists()), _LCA_CANDIDATES[0]))))
OUT_DIR        = DATA_ROOT / "ihc_lr"   # protected data root; aggregate tables + text-free note audit

## Load inputs

In [ ]:
scores = nsl.load_transfer_scores(SCORES_PATH, family="xgboost_cox", config="both", endpoint="nepc")
scores = scores[scores["landmark_days"].isin(LANDMARKS)]
anchors = il.load_anchor_dates(DATA_PATH)
members = nsl.cohort_members({c: None if c == "all" else cp.cohort_mrn_list_path(c) for c in COHORTS})
scored_ids = set(scores[il.ID_COL].astype(int))

lca = il.import_lca(LCA_ROOT)     # read-only import; nothing in LCA is written
pats = il.build_patterns(lca["ne_gate"].NE_MARKER.pattern)
print(f"{len(scored_ids):,} scored patients; landmarks {sorted(scores['landmark_days'].unique())}")
il.validate_anchor_coverage(scores, anchors)
print(f"LCA {il.lca_commit(LCA_ROOT)[:12]} · {lca['ne_gate'].NE_GATE_VERSION} · {il.IHC_REGEX_VERSION}")

## Tag IHC notes and merge the gated LLM findings

The regex tags every pathology report of a scored patient. The gated LLM findings are then joined on the same report.

In [ ]:
notes = il.load_pathology_notes(PATHOLOGY_PATH, mrns=scored_ids)
ihc_notes = il.tag_ihc_notes(notes, pats)
del notes   # drop report text; nothing below needs it

gated_notes, review_status = il.load_gated_ne_notes(lca, LLM_NEPC_DIR, mrns=scored_ids)
merged = il.add_days_from_adt(il.merge_ne_positive(ihc_notes, gated_notes, review_status), anchors)

print(f"{len(merged):,} pathology reports for scored patients "
      f"({merged['note_date'].isna().sum():,} undated, {merged['days_from_adt'].isna().sum():,} without days from ADT)")
print(f"  with IHC (any specimen): {merged['ihc_any'].sum():,}")
print(f"  qualifying IHC:          {merged['qualifies'].sum():,}")
print(f"  with a gated LLM finding: {merged['ne_gated'].sum():,} "
      f"(of {len(gated_notes):,} gated pathology reports for scored patients)")
print(f"  NE-positive qualifying:  {merged['ne_positive'].sum():,}")
display(pd.crosstab(merged["ihc_any"].map({True: "IHC", False: "no IHC"}).rename("regex"),
                    [merged["ne_gated"].map({True: "gated", False: "not gated"}).rename("LLM"),
                     merged["qualifies"].rename("qualifies")]))

## LR frame and attrition

In this table, `n_ne_gated_ihc_lost_to_specimen` counts patients whose future IHC report has a gated NE finding but fails the specimen rule, for example a metastasis biopsy that never states prostatic origin. `n_ne_gated_without_ihc_future` counts patients with a future gated pathology report that has no IHC regex hit. Neither group counts as NE-positive. `n_ne_pos_future` and `n_prior_ne_pos` count only patients eligible by LLM review status; `n_ne_review_excluded` counts those omitted from both NE fits.

In [ ]:
frame = il.build_ihc_frame(scores, merged, LANDMARKS, anchors=anchors, review_status=review_status)
models, coefficients, counts, predictions = il.run_ihc_lr(
    frame, members, LANDMARKS, min_events=MIN_EVENTS, n_folds=N_FOLDS,
    n_bootstrap=N_BOOTSTRAP, seed=SEED, return_predictions=True)
display(counts.set_index(["cohort", "landmark_days"]).T)

## Write outputs

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, df in (("lr_models", models), ("lr_coefficients", coefficients), ("lr_counts", counts),
                 ("lr_oof_predictions", predictions)):
    df.to_csv(OUT_DIR / f"{name}.csv", index=False)
merged.reindex(columns=il.AUDIT_COLUMNS).to_parquet(OUT_DIR / "ihc_notes_audit.parquet", index=False)
meta = {
    "lca_root": str(LCA_ROOT), "lca_commit": il.lca_commit(LCA_ROOT),
    "ne_gate_version": lca["ne_gate"].NE_GATE_VERSION, "ihc_regex_version": il.IHC_REGEX_VERSION,
    "ne_review_complete_statuses": sorted(il.NE_REVIEW_COMPLETE),
    "report_identity_policy": "reject_ambiguous_ids",
    "anchor_policy": "require_all_scored_patients",
    "scores_path": str(SCORES_PATH), "pathology_path": str(PATHOLOGY_PATH), "llm_nepc_dir": str(LLM_NEPC_DIR),
    "cohorts": list(COHORTS), "landmarks": list(LANDMARKS), "max_followup_days": il.MAX_FOLLOWUP_DAYS,
    "oof_predictions": "risk-only LR, stratified K-fold, one row per patient x landmark x outcome",
    "min_events": MIN_EVENTS, "n_folds": N_FOLDS, "n_bootstrap": N_BOOTSTRAP, "seed": SEED,
}
(OUT_DIR / "run_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Wrote lr_models.csv, lr_coefficients.csv, lr_counts.csv, lr_oof_predictions.csv "
      f"({len(predictions):,} patient rows), ihc_notes_audit.parquet, run_meta.json to {OUT_DIR}")

## Results

These show the odds ratio per SD of the risk score (95% CI) and Wald p, then the out-of-fold AUC (95% CI), per landmark × outcome, and calibration of the out-of-fold probabilities.

In [ ]:
def fmt(est, lo, hi, digits=2):
    return pd.Series([f"{e:.{digits}f} ({l:.{digits}f}–{h:.{digits}f})" if pd.notna(e) else ""
                      for e, l, h in zip(est, lo, hi)], index=est.index)

idx = ["cohort", "landmark_days"]
cols = list(il.OUTCOMES)
co = coefficients.assign(OR=lambda d: fmt(d["odds_ratio"], d["or_lo"], d["or_hi"]))
print("=== Odds ratio per SD of risk score (95% CI) ===")
display(co.pivot_table(index=idx, columns="outcome", values="OR", aggfunc="first").reindex(columns=cols))
print("=== Wald p ===")
display(co.pivot_table(index=idx, columns="outcome", values="p_wald", aggfunc="first").reindex(columns=cols)
          .apply(lambda col: col.map(lambda p: f"{p:.2g}" if pd.notna(p) else "")))

mo = models.assign(AUC=lambda d: fmt(d["auc"], d["auc_lo"], d["auc_hi"], 3))
print("=== Out-of-fold AUC (95% CI) ===")
display(mo.pivot_table(index=idx, columns="outcome", values="AUC", aggfunc="first").reindex(columns=cols))
mo = mo.assign(AUC_raw=lambda d: fmt(d["auc_raw_score"], d["auc_raw_score_lo"], d["auc_raw_score_hi"], 3))
print("=== AUC of the raw risk score (95% CI), no fitting ===")
display(mo.pivot_table(index=idx, columns="outcome", values="AUC_raw", aggfunc="first").reindex(columns=cols))
display(mo.pivot_table(index=idx, columns="outcome", values=["n", "events"], aggfunc="first"))

print("=== Markers on qualifying IHC notes of scored patients (any date) ===")
display(il.marker_frequency(merged))

model_notes = models[models["note"].fillna("").ne("")]
if len(model_notes):
    print("\nModels with notes (not fit, separation, or CV failure):")
    display(model_notes[["cohort", "landmark_days", "outcome", "n", "events", "n_failed_folds", "note"]])

pred_idx = ["cohort", "landmark_days", "outcome"]
print("=== Out-of-fold calibration of the risk-score LR (per fit) ===")
display(nsl.calibration_summary(predictions, pred_idx).set_index(pred_idx).round(4))
print("=== Observed vs mean OOF probability by prediction quintile ===")
display(nsl.calibration_bins(predictions, pred_idx, n_bins=5)
          .pivot_table(index=[*pred_idx, "bin"], values=["n", "events", "mean_predicted", "observed_rate"])
          .round(4))
